# Colab G4 · 论文参数对齐的 VRPO + CentralCritic

固定使用 AtaraxosAI/doudizhu 的官方提交 `acdcf199cf422e2b683cb22142b8e7cb33b14460`，调用原有 VRPOTrainer，不修改算法或降低精度。训练设置依据 [VRPO 附录 E](https://arxiv.org/html/2605.19235v1#A5) 及 Nature 斗地主补充材料。仅训练基础策略，不包含 belief 训练与在线搜索，也不宣称复现完整 Ataraxos 牌力。

Google Drive 文件夹、新训练/续训、可调训练时长、默认每30分钟保存和保留最近10份检查点功能保留。保存与停止在完整训练轮边界执行。


## 固定训练设置

- rollout=512；每轮 collect=2048；4 个 minibatch，每份512条轨迹。
- replay=64×2048=131,072条轨迹；完整放在CPU内存，原有随机无放回采样与数据类型不变。CPU/GPU随机数序列不同，不保证逐位复现。
- EMA=0.999；学习率与clip在5000轮后衰减；正则化在500轮后衰减。
- actor/critic各4 epochs；Muon/AdamW，基础学习率均4e-4；λ=0.95，γ=1；clip=0.02，正则系数0.1，梯度裁剪1。
- 总训练上限40,000轮；同时遵守下方时间限制，到达任一上限保存并结束。默认120分钟是便利设置，远少于论文完整训练预算。

这些参数不参与吞吐搜索。CPU线程为性能调优项。完整replay约59.46 GiB，启动前要求可用主机内存超过100 GiB。不要为避免OOM而直接缩小collect/replay或增大minibatches。


## G4 实测结果（论文参数约束，2026-10-11）

选定CPU线程 **8**。完整replay双种子复测的合并吞吐为 **175.659轨迹/秒**、12919.2环境步/秒、8554.0实际决策步/秒。

全程NVML显存峰值 **52.274 GiB**，保守显存余量 **43.318 GiB**；最低可用主机内存 **111.403 GiB**。硬件为RTX PRO6000 Blackwell Server Edition，PyTorch2.11.0+cu130。

3组线程初筛（16热身+8计量）；选定配置以seed0/1各运行64热身+16计量，确保replay已填满131072条并经过覆盖写入。所有合格区间无新增编译，训练指标有限。计量不含编译、保存和评测，不能代表端到端含Drive保存的速度。

线程间小差异不证明统计显著优势；这是有限候选内选择，不是全局最优或牌力证明。旧版175.304轨迹/秒使用不同学习参数，不作同条件比较。

实测训练未保存checkpoint。G4已关闭。GPU作业共51.01分钟；账户余额读数差7.12单位，不是精确逐作业账单。

| 阶段 | CPU线程 | 轨迹/秒 | 显存峰值GiB | 保守余量GiB |
|---|---:|---:|---:|---:|
| screen-00 | 6 | 163.354 | 52.280 | 43.312 |
| screen-01 | 4 | 163.246 | 52.216 | 43.377 |
| screen-02 | 8 | 164.043 | 52.274 | 43.318 |
| verify-0-seed0 | 8 | 175.400 | 52.274 | 43.318 |
| verify-0-seed1 | 8 | 175.919 | 51.981 | 43.611 |

In [ ]:
# @title 1. 用户配置
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

DRIVE_FOLDER = "/content/drive/MyDrive/doudizhu" # @param {type:"string"}
# 续训时改为上次运行打印的 RUN_NAME，并设置 RESUME_FROM="latest"。
RUN_NAME = datetime.now(ZoneInfo("Asia/Hong_Kong")).strftime("vrpo-g4-%Y%m%d-%H%M%S")
TRAINING_MINUTES = 120 # @param {type:"number"}
SAVE_EVERY_MINUTES = 30 # @param {type:"number"}
KEEP_CHECKPOINTS = 10 # @param {type:"integer"}
RESUME_FROM = "" # @param {type:"string"}
# 空字符串：新训练；"latest"：当前运行目录最新检查点；也可填完整 Drive 文件路径。
DISCONNECT_WHEN_DONE = True # @param {type:"boolean"}
SEED = 0 # @param {type:"integer"}
BEST = {'rollout': 512, 'collect': 2048, 'mini_batches': 4, 'threads': 8}
# threads 是实际使用的CPU线程数；设置1时原仓库会扩展为所有可用CPU。
COMMIT = "acdcf199cf422e2b683cb22142b8e7cb33b14460"
REPOSITORY = "https://github.com/AtaraxosAI/doudizhu.git"
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in (".", "..")
assert Path(DRIVE_FOLDER).is_absolute()
print("本次运行目录:", Path(DRIVE_FOLDER) / RUN_NAME)


In [ ]:
# @title 2. 挂载 Google Drive、读取目录及检查点
from google.colab import drive
import re

drive.mount("/content/drive")
DRIVE_ROOT = Path(DRIVE_FOLDER)
assert DRIVE_ROOT.resolve().is_relative_to(Path("/content/drive")), "请选择已挂载的 Google Drive 内的目录"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = DRIVE_ROOT / RUN_NAME / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Drive 文件夹内容（最多20项）:")
for entry in sorted(DRIVE_ROOT.iterdir())[:20]:
    print(entry.name)
existing = sorted((p for p in CHECKPOINT_DIR.glob("ckpt_*.pt") if p.is_file() and re.fullmatch(r"ckpt_\d+\.pt", p.name)),
                  key=lambda p: int(p.stem.split("_")[1]))
print("当前运行检查点:", [p.name for p in existing])
print("检查点目录:", CHECKPOINT_DIR)


In [ ]:
# @title 3. 安装固定版本训练代码（保留 Colab CUDA/PyTorch）
import os, subprocess, sys, json, math

assert math.isfinite(TRAINING_MINUTES) and TRAINING_MINUTES > 0
assert math.isfinite(SAVE_EVERY_MINUTES) and SAVE_EVERY_MINUTES > 0
assert isinstance(KEEP_CHECKPOINTS, int) and not isinstance(KEEP_CHECKPOINTS, bool) and KEEP_CHECKPOINTS >= 1
os.environ.update(WANDB_DISABLED="true", PYTORCH_ALLOC_CONF="expandable_segments:True",
    TORCHINDUCTOR_CACHE_DIR="/content/vrpo-inductor-cache", TORCHINDUCTOR_COMPILE_THREADS="2",
    OMP_NUM_THREADS=str(BEST["threads"]), MKL_NUM_THREADS=str(BEST["threads"]), NANOPOKERLIB_BUILD_JOBS="4")
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], text=True)
print(gpu)
assert "RTX PRO 6000" in gpu and "Blackwell" in gpu, "请先选择 G4 GPU 运行时"
free_mib = int(subprocess.check_output(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"], text=True).strip())
assert free_mib >= 80 * 1024, "启动前需至少80 GiB空闲显存，请重启运行时清除其他模型占用"
mem = dict(line.split(':', 1) for line in Path('/proc/meminfo').read_text().splitlines())
assert int(mem['MemAvailable'].split()[0]) / 2**20 > 100, "完整CPU replay需要超过100 GiB可用主机内存，请选择高内存G4"
REPO = Path("/content/doudizhu-vrpo-" + COMMIT[:12])
if not REPO.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(REPO)], check=True)
    subprocess.run(["git", "checkout", "--detach", COMMIT], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == COMMIT
assert not subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO, text=True).strip(), "训练代码已修改，请使用干净目录"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmake", "ninja", "pybind11", "tyro", "numpy>=2", "wheel", "setuptools"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "--no-deps", "-e", str(REPO / "nanopokerlib")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)], check=True)
print(subprocess.check_output([sys.executable, "-c", "import torch; print('torch:', torch.__version__, 'CUDA:', torch.version.cuda)"], text=True))


下一格只添加时间控制和 Drive 保存规则，训练本身继续调用原仓库算法。先在本地临时文件写完，再复制到 Drive 的 `.partial` 文件，完整复制后才公布文件名并清理旧检查点；失败时保留旧检查点。仅清理当前 `RUN_NAME/checkpoints` 中符合 `ckpt_数字.pt` 的文件，其他文件不动。


In [ ]:
%%writefile /content/vrpo_timed_training.py
"""Colab wall-time/Drive adapter; VRPO and checkpoint contents remain upstream."""
import json
import math
import os
from pathlib import Path
import re
import shutil
import signal
import sys
import tempfile
import time


def checkpoints(directory):
    return sorted((p for p in Path(directory).glob('ckpt_*.pt')
                   if p.is_file() and re.fullmatch(r'ckpt_\d+\.pt', p.name)),
                  key=lambda p: int(p.stem.split('_')[1]))


def publish_checkpoint(manager, state, directory, keep):
    if not isinstance(keep, int) or isinstance(keep, bool) or keep < 1:
        raise ValueError('keep_checkpoints must be a positive integer')
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f'ckpt_{state.iteration}.pt'
    if target.exists():
        raise FileExistsError(f'Refusing to overwrite {target}')
    pending = directory / (target.name + '.partial')
    try:
        with tempfile.TemporaryDirectory(prefix='vrpo-checkpoint-') as tmp:
            local = Path(tmp) / target.name
            manager.save_from_state(state, checkpoint_path=local, upload_to_wandb=False)
            shutil.copyfile(local, pending)
            if local.stat().st_size != pending.stat().st_size:
                raise IOError('Incomplete checkpoint copy to Drive')
            os.replace(pending, target)
    finally:
        pending.unlink(missing_ok=True)
    # Only prune this run's checkpoints after the new file is fully published.
    for old in checkpoints(directory)[:-keep]:
        old.unlink()
    return target


def timed_workflow(settings):
    import torch
    from marvel.training.workflow import TrainingWorkflow
    for key in ('training_minutes', 'save_every_minutes'):
        if not math.isfinite(settings[key]) or settings[key] <= 0:
            raise ValueError(f'{key} must be positive and finite')
    keep = settings['keep_checkpoints']
    if not isinstance(keep, int) or isinstance(keep, bool) or keep < 1:
        raise ValueError('keep_checkpoints must be a positive integer')

    class TimedWorkflow(TrainingWorkflow):
        def __init__(self, cfg):
            super().__init__(cfg, game=cfg.game, checkpoint_dir=settings['checkpoint_dir'])
            print(f'CPU threads: requested={settings["threads"]}, effective={torch.get_num_threads()}', flush=True)
            self.logger.progress_bar.total = None
            self.stop_requested = False
            self.last_saved_iteration = 0
            self.last_saved_time = time.monotonic()
            self.previous_handlers = {}

        def load_checkpoint_payload(self, checkpoint_path, *, device):
            # Stage on CPU so loading does not double the live GPU model/optimizer memory.
            return super().load_checkpoint_payload(checkpoint_path, device=torch.device('cpu'))

        def checkpoint_due(self):
            return time.monotonic() - self.last_saved_time >= settings['save_every_minutes'] * 60

        def save_checkpoint(self, state):
            if state.iteration == self.last_saved_iteration:
                return
            path = publish_checkpoint(self.checkpoint_manager, state, self.checkpoint_dir, keep)
            self.last_saved_iteration = state.iteration
            self.last_saved_time = time.monotonic()
            print(f'DRIVE_CHECKPOINT {path}', flush=True)

        def iterations(self):
            start = time.monotonic()
            self.last_saved_time = start
            def request_stop(signum, frame):
                self.stop_requested = True
                print('Stop requested; finish current iteration and save.', flush=True)
            for sig in (signal.SIGINT, signal.SIGTERM):
                self.previous_handlers[sig] = signal.signal(sig, request_stop)
            try:
                for iteration in super().iterations():
                    yield iteration
                    if self.stop_requested or time.monotonic() - start >= settings['training_minutes'] * 60:
                        self.save_checkpoint(self._state)
                        print(f'TRAINING_COMPLETE iteration={iteration} elapsed_seconds={time.monotonic()-start:.1f}', flush=True)
                        break
                else:
                    if self._state is not None and self.current_iteration is not None:
                        self.save_checkpoint(self._state)
            finally:
                for sig, handler in self.previous_handlers.items():
                    signal.signal(sig, handler)
    return TimedWorkflow


def validate_resume_payload(payload):
    required = {'agent_resume_state_dict', 'agent_state_dict', 'critic_state_dict',
                'agent_optimizer_state_dict', 'critic_optimizer_state_dict',
                'iteration', 'train_config'}
    missing = required - payload.keys()
    if missing or payload.get('train_config', {}).get('algo_name') != 'vrpo':
        raise ValueError(f'Expected a full VRPO training checkpoint; missing keys: {sorted(missing)}')


def main(settings):
    import torch
    from marvel.config import TrainConfig
    from marvel.training.vrpo import VRPOTrainer
    from marvel.utils import configure_torch
    torch.set_num_threads(settings['threads'])
    torch.set_num_interop_threads(1)
    directory = Path(settings['checkpoint_dir'])
    previous = checkpoints(directory)
    resume = settings.get('resume_from', '')
    if resume == 'latest':
        if not previous:
            raise FileNotFoundError(f'No checkpoints in {directory}')
        resume = str(previous[-1])
    elif previous and not resume:
        raise ValueError('Run folder already contains checkpoints: set resume_from="latest" or choose a new run folder')
    if resume and not Path(resume).is_file():
        raise FileNotFoundError(resume)
    if resume:
        payload = torch.load(resume, map_location='cpu', weights_only=True)
        validate_resume_payload(payload)
        if previous and int(payload['iteration']) < int(previous[-1].stem.split('_')[1]):
            raise ValueError('Resuming an older checkpoint requires a new run folder to preserve newer checkpoints')
        del payload
    cfg = TrainConfig(algo_name='vrpo', device='cuda', total_iterations=40_000,
        checkpoint_interval=-1, evaluation_interval=-1, param_stats_interval=-1,
        rollout_batch_size=settings['rollout'], collect_batch_size=settings['collect'],
        mini_batches=4, replay_buffer_scale=64,
        lr_stable_iterations=5000, reg_stable_iterations=500, agent_merging_beta=0.999,
        resume_from=resume or None, resume_optimizer=True, seed=settings.get('seed', 0),
        logging_dir=settings['logging_dir'])
    cfg.vrpo.replay_buffer_device = "cpu"
    configure_torch(cfg)
    VRPOTrainer(cfg, runtime_factory=timed_workflow(settings)).run()


if __name__ == '__main__':
    main(json.loads(Path(sys.argv[1]).read_text()))


In [ ]:
# @title 5. 启动训练，到时保存并结束
import signal

settings = dict(BEST, training_minutes=TRAINING_MINUTES, save_every_minutes=SAVE_EVERY_MINUTES,
    keep_checkpoints=KEEP_CHECKPOINTS, resume_from=RESUME_FROM, checkpoint_dir=str(CHECKPOINT_DIR),
    logging_dir=str(Path("/content/vrpo-training-logs") / RUN_NAME), seed=SEED)
settings_path = Path("/content/vrpo-training-settings.json")
settings_path.write_text(json.dumps(settings, indent=2))
(DRIVE_ROOT / RUN_NAME / "training-settings.json").write_text(json.dumps(dict(settings, commit=COMMIT), indent=2))
env = dict(os.environ, PYTHONPATH=str(REPO))
try:
    process = subprocess.Popen([sys.executable, "/content/vrpo_timed_training.py", str(settings_path)], cwd=REPO, env=env)
    try:
        returncode = process.wait()
    except KeyboardInterrupt:
        print("请求在当前完整训练轮结束后保存，请等待；强制删除运行时无法触发保存。")
        process.send_signal(signal.SIGINT)
        returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f"训练失败，退出码 {returncode}；请检查日志。旧检查点保留。将按配置同步Drive并释放资源。")
    print("训练已结束，检查点:")
    for path in sorted(CHECKPOINT_DIR.glob("ckpt_*.pt"), key=lambda p: int(p.stem.split("_")[1])):
        print(path, f"{path.stat().st_size / 2**20:.1f} MiB")
    print("继续训练时使用 RUN_NAME =", repr(RUN_NAME), "以及 RESUME_FROM = 'latest'")
finally:
    if DISCONNECT_WHEN_DONE:
        from google.colab import drive, runtime
        drive.flush_and_unmount(timeout_ms=120000)
        runtime.unassign()


In [ ]:
# @title 6. 手动释放 G4（失败或提前结束后也应执行）
from google.colab import drive, runtime
drive.flush_and_unmount(timeout_ms=120000)
runtime.unassign()


显存余量以本轮实测为依据，无法保证所有软件版本和长期运行状态都不OOM。若显存不足，先停止并检查运行时和配置；本版不通过缩小采集量、replay、模型或训练epoch换取速度。

Google Drive 首次挂载需要在 Colab 浏览器授权。默认每30分钟保存、保留最近10份均可调整。已有旧版检查点可以续训，但此前训练历史无法被新参数追溯修正；若要从头对齐论文，请使用新的 RUN_NAME。检查点包含策略、EMA策略、CentralCritic、优化器和轮数，原仓库不保存replay、环境与随机数状态，因此续训不等于不中断训练，replay会重新积累64轮。

强制关闭实例或无法完成当前轮时，只能恢复最近一次已发布检查点。Drive同步失败时不会自动释放，以便处理未同步文件。
